In [1]:
%pip install -qU langchain-pinecone pinecone-notebooks
%pip install --upgrade --quiet langchain-text-splitters tiktoken
%pip install langchain-openai
%pip install datasets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")

In [3]:
#retrieve dataframe
import pandas as pd
dataframe = pd.read_pickle("dataframe.pkl")

print(dataframe["text"])


0        so planets become more interesting moons
1             become places to go and revisit but
2           there was a whole other goal and that
3             was the search for intelligent life
4           still is in the universe oh man it is
                           ...                   
77209     All right. This has been Star Talk, the
77210                             Einstein Crumbs
77211       edition. Neil deGrasse Tyson here. As
77212       always, I bid you to keep looking up.
77213                                     [Music]
Name: text, Length: 77214, dtype: object


In [4]:
# create a new dataframe only with text
docs = dataframe["text"].tolist()
sources = dataframe["source_file"].tolist()

# optionally chunk large documents or sentences
from langchain.text_splitter import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=512, chunk_overlap=50)

splits = text_splitter.create_documents(docs)

# Attach metadata separately
for i, doc in enumerate(splits):
    doc.metadata = {"source_file": sources[i]}  # minimal metadata


In [5]:
# 1. Setup
import pandas as pd
from langchain.embeddings import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore  # ✅ NEW correct import
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
import os
from getpass import getpass

# Load environment variables
_ = load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or getpass("Enter your Pinecone API key: ")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("Enter your OpenAI API key: ")

# Load dataframe and take a small sample
df = pd.read_pickle("dataframe.pkl").head(100)  # Use only the first 100 rows
print(f"✅ Loaded dataframe with {len(df)} rows")
print(df.head(2))  # preview first 2 rows

# Prepare documents and sources
texts = df["text"].tolist()
sources = df["source_file"].tolist()
print(f"✅ Prepared {len(texts)} texts and sources")

# 2. Split texts and attach metadata
splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=20)
documents = [Document(page_content=t, metadata={"source_file": s}) for t, s in zip(texts, sources)]
print(f"✅ Created {len(documents)} Document objects")

split_docs = splitter.split_documents(documents)
print(f"✅ After splitting, got {len(split_docs)} document chunks")
print(f"Sample chunk content:\n{split_docs[0].page_content}")
print(f"Sample chunk metadata:\n{split_docs[0].metadata}")

# 3. Initialize Pinecone v3
index_name = "youtube-transcripts"
dimension = 1536

pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index if needed
if index_name not in [index["name"] for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"✅ Created index: {index_name}")
else:
    print(f"✅ Index '{index_name}' already exists.")

# Connect to the Pinecone Index
index = pc.Index(index_name)
print(f"✅ Connected to Pinecone index: {index_name}")

# Create OpenAI embedding instance
embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
print("✅ Initialized OpenAI embeddings")

# Use langchain_pinecone to store documents
vectordb = PineconeVectorStore.from_documents(
    documents=split_docs,
    embedding=embedding,
    index_name=index_name,
    batch_size=50  # limit batch size to reduce request payload size
)
print(f"✅ Stored {len(split_docs)} documents in Pinecone")

# 4. Set up retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 5})
print("✅ Retriever initialized")

# 5. Memory & Tools
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

llm = ChatOpenAI(temperature=0, model="gpt-4", openai_api_key=OPENAI_API_KEY)
print("✅ Initialized LLM")

# Define RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)
print("✅ RetrievalQA chain created")

# Define function for answering questions with sources
def answer_with_sources(input_text: str):
    print(f"\n📝 Query: {input_text}")
    result = qa_chain({"query": input_text})
    answer = result["result"]
    sources = list(set(doc.metadata.get("source_file", "Unknown") for doc in result["source_documents"]))
    print(f"🗒️ Retrieved {len(sources)} source documents")
    return f"{answer}\n\nSources:\n" + "\n".join(sources)


/opt/anaconda3/envs/sftenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded dataframe with 100 rows
                                       text  \
0  so planets become more interesting moons   
1       become places to go and revisit but   

                                         source_file  
0  40 - Neil deGrasse Tyson and Bill Nye Catch Up...  
1  40 - Neil deGrasse Tyson and Bill Nye Catch Up...  
✅ Prepared 100 texts and sources
✅ Created 100 Document objects
✅ After splitting, got 100 document chunks
Sample chunk content:
so planets become more interesting moons
Sample chunk metadata:
{'source_file': '40 - Neil deGrasse Tyson and Bill Nye Catch Up.en.txt'}
✅ Index 'youtube-transcripts' already exists.
✅ Connected to Pinecone index: youtube-transcripts


/var/folders/hq/l56ghxv518j9wg6pgqkbbvd80000gn/T/ipykernel_24238/2273493699.py:60: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)


✅ Initialized OpenAI embeddings
✅ Stored 100 documents in Pinecone
✅ Retriever initialized
✅ Initialized LLM
✅ RetrievalQA chain created


/var/folders/hq/l56ghxv518j9wg6pgqkbbvd80000gn/T/ipykernel_24238/2273493699.py:82: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature=0, model="gpt-4", openai_api_key=OPENAI_API_KEY)


In [6]:
from langsmith import traceable
import os
!pip install -U langsmith openai

# Set LangSmith env vars
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "LANGCHAIN_API_KEY"
os.environ["LANGCHAIN_PROJECT"] = "youtube-rag"

from langchain.agents import initialize_agent

# Define tool
tools = [
    Tool(
        name="YouTubeTranscriptQA",
        func=answer_with_sources,
        description="Useful for answering questions about YouTube video transcripts. Input should be a fully formed question."
    )
]
print("✅ Tools defined")

# Conversation memory
memory = ConversationBufferMemory(memory_key="chat_history")
print("✅ Conversation memory initialized")

# 6. Initialize Agent with tools and memory
agent = initialize_agent(
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    tools=tools,
    llm=llm,
    verbose=True,
    memory=memory,
    max_iterations=3
)
print("✅ Agent initialized and ready")


@traceable(name="YouTube RAG Trace")
def run_agent():
    agent.run("Your question here")

result = run_agent()



✅ Tools defined
✅ Conversation memory initialized
✅ Agent initialized and ready


> Entering new AgentExecutor chain...


/var/folders/hq/l56ghxv518j9wg6pgqkbbvd80000gn/T/ipykernel_24238/3092420209.py:23: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history")
/var/folders/hq/l56ghxv518j9wg6pgqkbbvd80000gn/T/ipykernel_24238/3092420209.py:27: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agen

Thought: Do I need to use a tool? No
AI: I'm sorry, but I can't provide a response without a specific question or topic. Could you please provide more details or ask a specific question?

> Finished chain.


In [ ]:
# Prompt templates for multi-query RAG system

personality_intro = "Answer as if you are Neil deGrasse Tyson, the astrophysicist known for being charismatic, cheeky, sometimes sarcastic, insightful, and eloquent."


prompt_templates = {
    "summary": "Please provide a concise summary of the following topic: '{}'",
    "source": "Please provide the video source for the following topic: '{}'",
    "explanation": "Explain in detail: '{}'",
    "compare": "Compare and contrast these two concepts: '{}' and '{}'",
    "timeline": "Give me a timeline of events related to: '{}'",
    "faq": "What are the most frequently asked questions about '{}', and their answers?",
    "key_points": "List the key points covered in: '{}'",
    "step_by_step": "Provide a step-by-step guide on how to: '{}'",
    "examples": "Give me examples related to '{}'",
    "pros_cons": "What are the pros and cons of '{}'",
    "common_mistakes": "What are the common mistakes people make regarding '{}', and how to avoid them?",
}

def ask_agent(agent, prompt_type, *args):
    if prompt_type not in prompt_templates:
        raise ValueError(f"Prompt type '{prompt_type}' not supported.")
    prompt = prompt_templates[prompt_type].format(*args)
    print(f"Prompt to model:\n{prompt}\n")
    response = agent.run(prompt)
    return response



In [12]:
# Example usage:
result = ask_agent(agent, "compare", "string theory", "pinpoint theory")
print(result)

Prompt to model:
Compare and contrast these two concepts: 'string theory' and 'pinpoint theory'



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? No
AI: As mentioned earlier, String theory is a theoretical framework in physics where the point-like particles of particle physics are replaced by one-dimensional objects called strings. It describes how these strings propagate through space and interact with each other. The key idea is that the fundamental constituents of reality are strings of energy, rather than point-like particles.

However, 'pinpoint theory' is not a recognized term in the field of physics. It's possible that there may be some confusion with the term. If you could provide more context or clarification about what you mean by 'pinpoint theory', I would be able to give a more accurate comparison.

> Finished chain.
As mentioned earlier, String theory is a theoretical framework in physics where the point-like particles of particle physics are repla

In [10]:
result = ask_agent(agent, "source", "Bill Nye")
print(result)

Prompt to model:
Please provide the video source for the following topic: 'Bill Nye'



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: YouTubeTranscriptQA
Action Input: What are some videos about 'Bill Nye'?
📝 Query: What are some videos about 'Bill Nye'?


/var/folders/hq/l56ghxv518j9wg6pgqkbbvd80000gn/T/ipykernel_24238/2273493699.py:96: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": input_text})


🗒️ Retrieved 1 source documents

Observation: I'm sorry, but the provided context does not contain information about any videos about 'Bill Nye'.

Sources:
40 - Neil deGrasse Tyson and Bill Nye Catch Up.en.txt
Thought:Do I need to use a tool? No
AI: Here is a video source related to 'Bill Nye': "Neil deGrasse Tyson and Bill Nye Catch Up". You can find this video on YouTube.

> Finished chain.
Here is a video source related to 'Bill Nye': "Neil deGrasse Tyson and Bill Nye Catch Up". You can find this video on YouTube.


In [15]:
result = ask_agent(agent, "summary", "the big bang")


Prompt to model:
Please provide a concise summary of the following topic: 'the big bang'



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? No
AI: The Big Bang theory is the prevailing cosmological model that describes the development of the Universe. According to this theory, the Universe began as a very hot, small, and dense superforce (with no stars, atoms, form, or structure), then expanded over a long period of time — about 13.8 billion years — to its current size and cooled throughout this expansion phase. This theory is supported by various forms of empirical evidence, including redshifts of distant galaxies, cosmic microwave background radiation, and the abundance of light elements in the universe.

> Finished chain.


In [16]:
#evaluate with BLEU
!pip install nltk
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt')  # download punkt tokenizer

def compute_bleu(reference_text, candidate_text):
    """
    Compute BLEU score between a reference answer and candidate answer.
    """
    reference_tokens = nltk.word_tokenize(reference_text.lower())
    candidate_tokens = nltk.word_tokenize(candidate_text.lower())
    smoothing = SmoothingFunction().method1
    score = sentence_bleu([reference_tokens], candidate_tokens, smoothing_function=smoothing)
    return score

# Update your answer function to accept a reference answer
def answer_with_bleu(input_text: str, reference_answer: str = None):
    print(f"\n📝 Query: {input_text}")
    result = qa_chain({"query": input_text})
    answer = result["result"]
    sources = list(set(doc.metadata.get("source_file", "Unknown") for doc in result["source_documents"]))
    print(f"🗒️ Retrieved {len(sources)} source documents")
    
    output = f"{answer}\n\nSources:\n" + "\n".join(sources)
    
    if reference_answer:
        bleu_score = compute_bleu(reference_answer, answer)
        print(f"🔵 BLEU score: {bleu_score:.4f}")
        output += f"\n\nBLEU score compared to reference: {bleu_score:.4f}"
        
    return output


[nltk_data] Downloading package punkt to /Users/test/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [19]:
query = "What would happen if we find evidence of life on another planet?"

reference = "it is\
very reasonable that maybe in my\
lifetime but in your kids is's lifetime\
somebody's going to find evidence of\
Life on another world and because if we\
found such a signal it would dare I say\
it change change the world the day we\
discover Life Will signal a change in\
the human condition that we cannot\
foresee."

response = answer_with_bleu(query, reference)
print(response)



📝 Query: What would happen if we find evidence of life on another planet?
🗒️ Retrieved 1 source documents
🔵 BLEU score: 0.0009
The context provided does not contain enough information to answer your question.

Sources:
40 - Neil deGrasse Tyson and Bill Nye Catch Up.en.txt

BLEU score compared to reference: 0.0009


In [24]:
#voice input
# !pip install openai-whisper pyaudio pyttsx3
!pip install sounddevice wavio



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sounddevice]


In [27]:
import whisper
import pyttsx3
import sounddevice as sd
import numpy as np
import wavio

# Initialize Whisper model
model = whisper.load_model("base")

# Initialize TTS engine
tts_engine = pyttsx3.init()

def record_audio(duration=5, fs=16000):
    print("🎙️ Recording...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()
    recording = np.squeeze(recording)
    wavio.write("user_input.wav", recording, fs, sampwidth=2)
    print("🎙️ Recording finished and saved as user_input.wav")

def transcribe_audio(filepath="user_input.wav"):
    print("📝 Transcribing audio with Whisper...")
    result = model.transcribe(filepath)
    text = result["text"]
    print(f"🗣️ Transcription: {text}")
    return text

def speak_text(text):
    print(f"🤖 Speaking: {text}")
    tts_engine.say(text)
    tts_engine.runAndWait()

def voice_rag_interaction():
    record_audio(duration=5)
    user_question = transcribe_audio()
    answer = answer_with_sources(user_question)  # your RAG QA function
    print(f"Answer:\n{answer}")
    speak_text(answer)

while True:
    voice_rag_interaction()
    cont = input("Ask another question? (y/n): ")
    if cont.lower() != "y":
        break


🎙️ Recording...
🎙️ Recording finished and saved as user_input.wav
📝 Transcribing audio with Whisper...


/opt/anaconda3/envs/sftenv/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🗣️ Transcription:  What is Bill Nye famous for?

📝 Query:  What is Bill Nye famous for?
🗒️ Retrieved 1 source documents
Answer:
Bill Nye is famous for being a science educator, particularly known for his television show "Bill Nye the Science Guy."

Sources:
40 - Neil deGrasse Tyson and Bill Nye Catch Up.en.txt
🤖 Speaking: Bill Nye is famous for being a science educator, particularly known for his television show "Bill Nye the Science Guy."

Sources:
40 - Neil deGrasse Tyson and Bill Nye Catch Up.en.txt
🎙️ Recording...
🎙️ Recording finished and saved as user_input.wav
📝 Transcribing audio with Whisper...


/opt/anaconda3/envs/sftenv/lib/python3.10/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🗣️ Transcription:  What will happen when we discover there's life?

📝 Query:  What will happen when we discover there's life?
🗒️ Retrieved 1 source documents
Answer:
The text suggests that the discovery of life will signal a change, but it does not provide specific details about what this change will be.

Sources:
40 - Neil deGrasse Tyson and Bill Nye Catch Up.en.txt
🤖 Speaking: The text suggests that the discovery of life will signal a change, but it does not provide specific details about what this change will be.

Sources:
40 - Neil deGrasse Tyson and Bill Nye Catch Up.en.txt
